# Kentucky — KRS insurance chapters (295–304) → `data/kentucky/ins_codes/*.md`

Kentucky’s **insurance-related** Revised Statutes on Justia span **KRS Chapters 295–302** (mortgage guaranty, division of insurance, company structures, reinsurance, etc.) and **Chapter 304 — Insurance Code** (subtitles **304.1–304.xx** with dense section numbering). The official portal is **[legislature.ky.gov — KRS](https://legislature.ky.gov/Law/Pages/KRS.aspx)**; this notebook mirrors **Justia**: year-scoped paths like **`/codes/kentucky/2024/chapter-304/…`**.

**Cloudflare** often blocks plain **`httpx`**; we use **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/indiana.ipynb`** / **`kansas.ipynb`**).

**Discovery:** BFS from the chapter index for each of **`295, 296, 297, 298, 299, 301, 302, 304`**, following only paths under **`/codes/kentucky/{CODE_YEAR}/chapter-{n}/`** that are not section pages; every matching **`section-(295|…|304)-…`** link is collected (on the order of **~800+** sections for 2024).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`KRS_sec_<label>.md`** where **`label`** is the URL segment after **`section-`**, hyphens → underscores (e.g. `KRS_sec_304_1_010.md`).

**Config:** **`CODE_YEAR`** seeds **`/codes/kentucky/{year}/chapter-…/`**. **`MAX_SECTIONS`** caps downloads (**0** = all). **`MAX_DISCOVERY_PAGES`** caps index fetches during discovery (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips discovery when **`_kentucky_insurance_section_urls.txt`** exists.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
CODE_YEAR = "2024"
INSURANCE_CHAPTERS = (295, 296, 297, 298, 299, 301, 302, 304)
CHAPTER_PREFIXES = tuple(f"/codes/kentucky/{CODE_YEAR}/chapter-{c}" for c in INSURANCE_CHAPTERS)
START_URLS = [f"{BASE}/codes/kentucky/{CODE_YEAR}/chapter-{c}/" for c in INSURANCE_CHAPTERS]

OUT_DIR = Path("data") / "kentucky" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_kentucky_insurance_section_urls.txt"
REUSE_DISCOVERED_URLS = True

section_label_re = re.compile(
    r"/section-((?:295|296|297|298|299|301|302|304)(?:-[0-9]+)+)/?$",
    re.I,
)


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def in_scope_index_path(p: str) -> bool:
    if f"/codes/kentucky/{CODE_YEAR}/" not in p:
        return False
    if "/section-" in p.lower():
        return False
    return any(p.startswith(pref) for pref in CHAPTER_PREFIXES)


def discover_section_urls() -> list[str]:
    """BFS chapter/subtitle index pages; collect insurance chapter section URLs."""
    seen: set[str] = set()
    in_q: set[str] = {path_key(u) for u in START_URLS}
    q: deque[str] = deque(START_URLS)
    sections: set[str] = set()
    fetches = 0

    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if not in_scope_index_path(pk):
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu)
            if f"/codes/kentucky/{CODE_YEAR}/" not in p:
                continue
            if "/section-" in p.lower():
                if section_label_re.search(p):
                    sections.add(BASE + p + "/")
            else:
                if not any(p.startswith(pref) for pref in CHAPTER_PREFIXES):
                    continue
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")

    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    m = section_label_re.search(urlparse(url).path)
    if not m:
        raise ValueError(f"cannot parse section id from {url!r}")
    return m.group(1)


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"KRS_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    drop_exact = {
        "of",
        "this Section",
        "Universal Citation:",
        "Next",
        "Previous",
    }
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if s in drop_exact:
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Kentucky Revised Statutes" in s:
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_insurance_chapters() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs (insurance chapters {INSURANCE_CHAPTERS})")
        all_urls = found
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"KRS {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Kentucky Revised Statutes — insurance chapters (295–304)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [legislature.ky.gov — KRS](https://legislature.ky.gov/Law/Pages/KRS.aspx)\n\n"
                    f"**Section (URL label):** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_insurance_chapters()


Discovered 861 section URLs (insurance chapters (295, 296, 297, 298, 299, 301, 302, 304))
… 200/861 (wrote=200 skipped=0 failed=0)
… 400/861 (wrote=400 skipped=0 failed=0)
… 600/861 (wrote=600 skipped=0 failed=0)
… 800/861 (wrote=800 skipped=0 failed=0)
Done. wrote=861 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/kentucky/ins_codes


{'wrote': 861, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
